# L63 · 音频分类模型 — CNN + Mel 特征，在 Speech Commands 上定义网络

**目标**：设计 CNN 分类器，输入 mel 特征图 `(B, 1, n_mels, T)`，输出 `n_classes` 类的 logit。

🔗 **Aurora 连接**：输入特征由 `aurora.audio.mel` 的 `mel_spectrogram()` 产出；本节是 Aurora 第一个完整的 ML 应用，将 DSP 特征与 PyTorch 模型直接打通。

Mel 频谱图是一张「图片」：横轴是时间帧，纵轴是 Mel 频段，像素值是能量。CNN 天然擅长在这类二维网格上提取局部模式——低层捕捉短时音素，高层组合成关键词。我们用两层卷积（convolution） + 全局平均池化（global average pooling，GAP）把可变长度的时间序列压缩成固定向量，再接全连接分类。

In [ ]:
import torch
import torch.nn as nn

## 1. 输入 Shape：`(B, 1, n_mels, T)`

PyTorch 的 `nn.Conv2d` 期望输入格式为 `(batch, channels, height, width)`。对 mel 特征图：

- `B`：batch size
- `1`：单通道（灰度图，对应单帧能量）
- `n_mels`：Mel 频段数，作为「高度」
- `T`：时间帧数，作为「宽度」

典型值：`n_mels=40`，1秒音频 @ 16kHz + hop=160 → `T=101`。注意 `T` 在推理时可变，模型必须对任意 `T` 都能运行，这是设计 global pooling 的动机。

In [ ]:
# 模拟一批 mel 特征图
B, n_mels, T = 8, 40, 81
x = torch.randn(B, 1, n_mels, T)
print(f'输入 shape: {x.shape}')  # torch.Size([8, 1, 40, 81])

## 2. 1D 时间卷积：`Conv2d(1, 32, kernel_size=(n_mels, 5))`

卷积核（convolution kernel）高度等于 `n_mels`（覆盖全部 Mel 维度），宽度为 5（滑动时间窗）。一次卷积后，频率维从 `n_mels` 压缩到 `1`，时间维保持（配合 `padding=(0,2)`）：

```
输入:  (B, 1,  n_mels, T)
conv1: (B, 32, 1,      T)   ← 频率维完全压缩，时间维不变
```

这等价于在时间轴上做 1D 卷积，但同时对所有 Mel 频段做加权求和。感受野（receptive field，RF）宽度 = kernel_width + (layers-1)×(kernel_width-1)，单层时就是 5 帧。

In [ ]:
# 验证 conv1 输出 shape
n_mels = 40
conv1 = nn.Conv2d(1, 32, kernel_size=(n_mels, 5), padding=(0, 2))
x = torch.randn(8, 1, n_mels, 81)
y = conv1(x)
print(f'conv1 输出 shape: {y.shape}')  # torch.Size([8, 32, 1, 81])
# 频率维 → 1，时间维 81 不变（padding=(0,2) 补偿 kernel宽5 的边缘）

## 3. Global Average Pooling：把时间维平均掉

分类任务需要固定长度的特征向量，但 `T` 在推理时可变。Global Average Pooling 对时间轴取平均：

```
(B, 64, 1, T) → mean(dim=-1) → (B, 64, 1) → squeeze() → (B, 64)
```

这比 Flatten + 全连接更鲁棒：参数量与 `T` 无关，且对时间平移有一定不变性。之后接 `nn.Linear(64, n_classes)` 输出每类的 logit，再配合 `CrossEntropyLoss` 训练。

In [ ]:
# 演示 Global Average Pooling
feat = torch.randn(8, 64, 1, 81)   # conv 输出
pooled = feat.mean(dim=-1).squeeze(dim=-1)  # (8, 64)
print(f'池化后 shape: {pooled.shape}')  # torch.Size([8, 64])

# 不同 T 得到相同输出 shape
feat2 = torch.randn(8, 64, 1, 150)
pooled2 = feat2.mean(dim=-1).squeeze(dim=-1)
print(f'T=150 池化后 shape: {pooled2.shape}')  # torch.Size([8, 64])

## 4. ✏️ 实现 `class KeywordCNN(nn.Module)`

签名：`__init__(self, n_mels=40, n_classes=10)`

**推理路线**：
1. `conv1`: `Conv2d(1, 32, (n_mels, 5), padding=(0,2))` + ReLU → 输出 `(B, 32, 1, T)`，频率维压缩为 1
2. `conv2`: `Conv2d(32, 64, 3, padding=1)` + ReLU + `MaxPool2d((1,2))` → 时间维减半 `(B, 64, 1, T//2)`
3. `global_avg_pool`：`mean(dim=-1).squeeze(dim=-1)` → `(B, 64)`
4. `fc`: `Linear(64, n_classes)` → `(B, n_classes)` logit

**参考输入输出**：
```python
m = KeywordCNN(40, 10)
x = torch.randn(8, 1, 40, 81)
out = m(x)
# out.shape == torch.Size([8, 10])
```

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
class KeywordCNN(nn.Module):
    def __init__(self, n_mels=40, n_classes=10):
        super().__init__()
        # ✏️ TODO: 定义 conv1, conv2, pool, fc
        pass

    def forward(self, x):
        # ✏️ TODO: 实现前向传播，返回 (B, n_classes) logit
        pass

In [ ]:
# 检查：输出 shape 必须为 (8, 10)
m = KeywordCNN(40, 10)
out = m(torch.randn(8, 1, 40, 81))
assert out.shape == (8, 10), f'期望 (8,10)，实际 {out.shape}'
print('✅ KeywordCNN 输出 shape 正确：', out.shape)

# 额外检查：不同 T 也能运行（Global Pooling 的核心优势）
out2 = m(torch.randn(4, 1, 40, 150))
assert out2.shape == (4, 10)
print('✅ 可变时间长度 T=150 也通过：', out2.shape)

## 5. 参数实验：时间卷积核大小对感受野的影响

**实验变量**：`conv1` 的时间维 kernel 宽度（3 vs 7）

单层感受野 = kernel_width；两层叠加后感受野 = k1 + k2 - 1。

```
kernel=3: conv1感受野=3帧, conv2感受野=3帧 → 总计 3+3-1=5帧
kernel=7: conv1感受野=7帧, conv2感受野=3帧 → 总计 7+3-1=9帧
```

**预期现象**：kernel=7 的模型对较长音素模式（如元音）响应更强，kernel=3 对短促辅音更敏感。实际验证需用真实语音数据，这里只比较参数量和激活范数。

In [ ]:
def make_cnn_with_k(k):
    """用不同时间 kernel 宽度构造 KeywordCNN（修改 conv1 的 kernel）"""
    n_mels = 40
    net = nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=(n_mels, k), padding=(0, k//2)),
        nn.ReLU(),
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d((1, 2)),
    )
    return net

x = torch.randn(8, 1, 40, 81)
for k in [3, 5, 7]:
    net = make_cnn_with_k(k)
    params = sum(p.numel() for p in net.parameters())
    out = net(x)
    # 用激活范数粗估响应强度
    norm = out.norm().item()
    receptive_field = k + 3 - 1  # conv1 + conv2 的感受野
    print(f'kernel={k}: 参数量={params:,}, 时间感受野={receptive_field}帧, 激活范数={norm:.2f}')

## 本课收束

`KeywordCNN` 通过 `conv1`（全频率压缩）+ `conv2`（特征精炼）+ Global Average Pooling，将任意长度的 mel 特征图映射为 `(B, n_classes)` logit 向量。输入特征直接来自 `aurora.audio.mel.mel_spectrogram()`，两者无缝对接，构成 Aurora 第一个端到端的 ML pipeline。下一节（M2-K3）将在 Google Speech Commands 子集上训练该模型，观察 loss 曲线与分类准确率。